In [24]:
import pandas as pd
import yaml

In [25]:
with open("../config.yaml", "r") as file:
    config = yaml.safe_load(file)

In [26]:
config

{'data': {'raw': {'file1': '../data/raw/df_final_demo.txt',
   'file2': '../data/raw/df_final_experiment_clients.txt',
   'file3': '../data/raw/df_final_web_data_pt_1.txt',
   'file4': '../data/raw/df_final_web_data_pt_2.txt',
   'file5': '../data/raw/final_web_data_merged.csv'},
  'clean': {'file1': '../data/clean/df_demo_cleaned.csv',
   'file2': '../data/clean/df_experiment_clients_cleaned.csv'}}}

In [29]:
df_w = pd.read_csv(config['data']['raw']['file3'], quotechar='"')
df_w = df_w.drop_duplicates()
df_w.head()

,client_id,visitor_id,visit_id,process_step,date_time
0,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:27:07
1,9988021,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:26:51
2,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:19:22
3,9988021,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:19:13
4,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:18:04


In [30]:
#find most freq client to get a sense of the data
top_client = df_w.client_id.value_counts().index[0]
df_w[(df_w.client_id==top_client) & (df_w.process_step == 'confirm')]
finished_ids = df_w[df_w.process_step == 'confirm'].client_id

In [31]:
def contains_step_error(lst, sub):
    n = len(sub)
    return any(lst[i:i+n] == sub for i in range(len(lst)-n+1))

# result = df_u[df_u['process_step'].apply(lambda x: contains_sublist(x, ['step_1', 'step_2', 'step_1']))]

In [32]:
# #distinct visit_ids
# visit_ids_u = df_w.visit_id.drop_duplicates()
# made_error_id =[]
# err_rate_s_1 = []
# err_rate_1_2 = []
# err_rate_2_3 = []
# err_rate_3_c = []
# for id in visit_ids_u:
#     df_u = df_w[df_w.visit_id==id] #.sort_values(by='date_time')
#     if sum(df_u.duplicated(subset=["process_step"]))>0:
#         step_list = df_u['process_step'].tolist()
#         if contains_step_error(step_list, ['start', 'step_1', 'start']):
#             err_rate_s_1.append('client_id')
#         if contains_step_error(step_list, ['step_1', 'step_2', 'step_1']):
#             err_rate_1_2.append('client_id')
#         if contains_step_error(step_list, ['step_2', 'step_3', 'step_2']):
#             err_rate_2_3.append('client_id')
#         if contains_step_error(step_list, ['step_3', 'confirm', 'step_3']):
#             err_rate_3_c.append('client_id')
        

In [33]:
# def trim_to_last_start(df_u):
#     start_times = df_u[df_u.process_step == 'start']['date_time']
#     if start_times.empty:
#         return df_u
#     last_start = start_times.max()
#     return df_u[df_u['date_time'] >= last_start]

# # df_clean = df_w.groupby('client_id', group_keys=False).apply(trim_to_last_start)
# df_clean = df_w.groupby('client_id', group_keys=False).apply(trim_to_last_start).reset_index()


In [35]:
patterns = {
    's_1': ['start', 'step_1', 'start'],
    '1_2': ['step_1', 'step_2', 'step_1'],
    '2_3': ['step_2', 'step_3', 'step_2'],
    '3_c': ['step_3', 'confirm', 'step_3']
}
results = {key: set() for key in patterns}

for client_id, group in df_w.groupby('client_id'):
    step_list = group['process_step'].tolist()
    for key, pattern in patterns.items():
        if contains_step_error(step_list, pattern):
            results[key].add(client_id)

total = df_w['client_id'].nunique()
error_rates = {key: len(ids) / total for key, ids in results.items()}
print(error_rates)

{'s_1': 0.11602815502389066, '1_2': 0.039269750475244475, '2_3': 0.06201298145262112, '3_c': 0.0025346371872377593}
